In [1]:
%matplotlib inline
__import__("os").environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

In [2]:
import os
import copy
import time
import warnings
import matplotlib.pyplot as plt

import torch
import random
import numpy as np

from pathlib import Path

seed=0
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

In [3]:
import sys
sys.path.append('../../..')

%load_ext autoreload
%autoreload 2

from computer_vision.yolov11.modules.detector import DetectionModel
from computer_vision.yolov11.modules.validator import DetectionValidator
from computer_vision.yolov11.modules.trainer import DetectionTrainer
from computer_vision.yolov11.parameter_parser import parser
from computer_vision.yolov11.utils.check import check_imgsz
from computer_vision.yolov11.data.dataset import YOLODataset
from computer_vision.yolov11.utils.plotting import plot_labels, plot_predictions


In [4]:
data_dirpath=r'D:/data/ultralytics/coco'
result_dirpath='D:/results/yolov11/training'

argument=f'''--root {data_dirpath} --train-image-dirname images/train2017 --train-label-dirname labels/train2017
--val-image-dirname images/val2017 --val-label-dirname labels/val2017
--data-cfg ../coco.yaml --hyperparam ../default.yaml --model-cfg ../yolo11.yaml 
--batch-size 16 --output-dirpath {result_dirpath} --checkpoint-dirpath {result_dirpath}/checkpoints 
--epochs 200 --time 6.5  --print-freq 150 --grad-clip 100''' # we want to see if we disable mosaic, do images having the same size for stacking?
# close_mosaic is set to 10, so it should stop mosaic at epoch 3
args=parser.parse_args(argument.split())

if not os.path.isdir(args.checkpoint_dirpath): os.makedirs(args.checkpoint_dirpath)

In [5]:
trainer=DetectionTrainer(args, cfg=args.hyperparam, inch=3)
trainer._setup_train()
trainer.epochs, trainer.start_epoch

Resume training with checkpoint directory set to D:/results/yolov11/training/checkpoints
In modules.trainer.DetectionTrainer.__init__ self.args.worker 2
In modules.trainer.DetectionTrainer._init__ self.args.resume  True
In module.trainer.DetectionTrainer._setup_train: Freezing layer model.23.dfl.conv.weight
In data.dataset.YOLODataset.update_images_labels cache path D:\data\ultralytics\coco\labels\train2017.cache exist. Load it!!!
Scanning D:\data\ultralytics\coco\labels\train2017.cache ... 117266 images with 1021 missing and 0 empty files as well as 0 corrupt files
In data.dataset.YOLODataset.__init__ max_buffer_length  128  ni  118287
In data.dataset.YOLODataset.update_images_labels cache path D:\data\ultralytics\coco\labels\val2017.cache exist. Load it!!!
Scanning D:\data\ultralytics\coco\labels\val2017.cache ... 4952 images with 48 missing and 0 empty files as well as 0 corrupt files
In data.dataset.YOLODataset.__init__ max_buffer_length  0  ni  5000
optimizer: "optimizer=auto" fou

(200, 200)

In [6]:
trainer._clear_memory(threshold=0.5) # prevent VRAM spike
# self.metrics, self.fitness=self.validate()
# metrics=self.validator(trainer=self, hyp=self.args)

In [7]:
hyp=trainer.args
trainer.validator.training=trainer is not None
validator=trainer.validator
validator.device=trainer.device
validator.data=trainer.data
model=trainer.model  # trainer.ema.ema or trainer.model
print("model ", model)
model=model.float()
validator.loss=torch.zeros(3,  device=trainer.device)#torch.zeros_like(trainer.loss_items, device=trainer.device)
#self.args.plots&=trainer.stopper.possible_stop or (trainer.epoch==trainer.epochs-1)
model.eval()

model  DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU()
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU()
    )
    (2): C3k2(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU()
      )
      (cv2): Conv(
        (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU()
      )
      (m): ModuleList(
        (0): Bottlene

DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU()
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU()
    )
    (2): C3k2(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU()
      )
      (cv2): Conv(
        (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU()
      )
      (m): ModuleList(
        (0): Bottleneck(
   

In [8]:
validator.init_metrics(model)
validator.jdict=[] # empty before each val

In [9]:
for batch_i, batch in enumerate(validator.dataloader):

    validator.batch_i=batch_i
    
    # Preprocessing
    batch=validator.preprocess(batch)

    # Inference
    with torch.no_grad(): preds=model(batch['img'])

    # Loss
    if validator.training:
        assert hyp is not None
        with torch.no_grad(): validator.loss+=model.loss(batch=batch, preds=preds, hyp=hyp)[1]
            
    # Postprocess  
    preds=validator.postprocess(preds) # with bounding boxes in xyxy in pixel units

    validator.update_metrics(preds, batch)

    if validator.args.plots and batch_i<3:
        fname='-'.join(Path(l).stem for l in batch['im_file'])
        copied_batch=copy.deepcopy(batch)
        copied_batch={k:(v.detach().cpu() if isinstance(v, torch.Tensor) else v) for k, v in copied_batch.items()}
        plot_labels(copied_batch, fname=validator.save_dir/'{}-val-label.jpg'.format(fname),
                    xywh=True)
        copied_preds=copy.deepcopy(preds)
        for i in range(len(copied_preds)):
            copied_preds[i]={k:(v.detach().cpu() if isinstance(v, torch.Tensor) else v) for k, v in copied_preds[i].items()}
        plot_predictions(images=batch['img'].clone().cpu(), prediction=copied_preds, 
                         fname=validator.save_dir/'{}-val-pred.jpg'.format(fname))


In [10]:
stats=validator.get_stats()
validator.finalize_metrics()
results={**stats, **trainer.label_loss_items(validator.loss.cpu()/len(validator.dataloader), prefix='val')}
results

{'metrics/precision(B)': 0.19561495266147735,
 'metrics/recall(B)': 0.12265417078787925,
 'metrics/mAP50(B)': 0.11411814351893086,
 'metrics/mAP50-95(B)': 0.06897294969247182,
 'fitness': 0.06897294969247182,
 'val/box_loss': 1.47195,
 'val/cls_loss': 1.5656,
 'val/dfl_loss': 1.39287}